In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Dict


In [ ]:
# --------------------------
# Global simulation constants
# --------------------------
DT = 0.05  # s

# FIA MGU-K constraints
MGUK_MAX_DEPLOY_POWER = 120_000.0   # W
MGUK_MAX_DEPLOY_ENERGY_LAP = 4e6    # J
MGUK_MAX_REGEN_POWER = 120_000.0    # W (early-stage assumption)

# Battery model
BATTERY_CAPACITY_J = 4e6            # J
SOC_MIN, SOC_MAX = 0.20, 1.00
INITIAL_SOC = 0.78

# Vehicle/track physics parameters (calibration targets for modern F1-like behavior)
VEHICLE_MASS = 798.0                # kg
G = 9.81                            # m/s^2
RHO_AIR = 1.225                     # kg/m^3
CD_A = 1.00                         # effective drag area
C_RR = 0.012                        # rolling resistance coefficient
TRACTION_FORCE_LIMIT = 17_500.0     # N
ICE_WHEEL_POWER = 650_000.0         # W
MAX_BRAKE_DECEL = 5.8 * G           # m/s^2 cap to avoid non-physical braking spikes
MIN_SPEED_MPS = 35.0 / 3.6          # m/s, prevent unrealistic near-stop behavior

# Hybrid conversion efficiencies
DEPLOY_EFFICIENCY = 0.95
REGEN_EFFICIENCY = 0.70

# --------------------------
# Phase 1 calibration target bands
# --------------------------
CAL_TARGETS = {
    "lap_time_min_s": 85.0,
    "lap_time_max_s": 140.0,
    "top_speed_min_kph": 280.0,
    "top_speed_max_kph": 360.0,
    "min_speed_min_kph": 45.0,
    "min_speed_max_kph": 190.0,
    "max_soc_balance_error": 0.02,
}


In [ ]:
@dataclass(frozen=True)
class TrackSegment:
    name: str
    segment_type: str  # "accel", "brake", "coast"
    length_m: float
    v_entry_kph: float
    v_exit_kph: float
    deployment_priority: float
    regen_priority: float
    overtaking_relevance: float
    notes: str

    @property
    def v_entry_mps(self) -> float:
        return self.v_entry_kph / 3.6

    @property
    def v_exit_mps(self) -> float:
        return self.v_exit_kph / 3.6


def build_gilles_villeneuve_track() -> List[TrackSegment]:
    return [
        TrackSegment("Start/Finish Straight", "accel", 780, 145, 305, 0.70, 0.05, 0.45, "Medium-value deployment before T1."),
        TrackSegment("Turn 1-2 Chicane Braking", "brake", 210, 305, 125, 0.05, 0.80, 0.40, "Heavy regen zone."),
        TrackSegment("T2 Exit to T3 Approach", "accel", 520, 125, 285, 0.60, 0.05, 0.25, "Moderate deployment value."),
        TrackSegment("Turn 3-4 Chicane Braking", "brake", 180, 285, 118, 0.05, 0.65, 0.20, "Moderate-heavy braking."),
        TrackSegment("Casino Straight", "accel", 820, 118, 298, 0.75, 0.05, 0.50, "Strong deployment value."),
        TrackSegment("L'Epingle Hairpin Braking (T10)", "brake", 220, 298, 86, 0.05, 1.00, 0.85, "Strongest regen + overtaking setup."),
        TrackSegment("Back Straight (T10 Exit to T13)", "accel", 1080, 86, 330, 1.00, 0.05, 1.00, "Primary deployment target."),
        TrackSegment("Wall of Champions Braking (T13-14)", "brake", 190, 330, 145, 0.05, 0.85, 0.70, "Heavy final braking."),
    ]


def validate_track(track: List[TrackSegment]) -> None:
    valid_types = {"accel", "brake", "coast"}
    if len(track) == 0:
        raise ValueError("Track cannot be empty.")

    for i, seg in enumerate(track):
        if seg.segment_type not in valid_types:
            raise ValueError(f"Invalid segment type in {seg.name}")
        if seg.length_m <= 0:
            raise ValueError(f"Invalid length in {seg.name}")
        for field_name, value in (
            ("deployment_priority", seg.deployment_priority),
            ("regen_priority", seg.regen_priority),
            ("overtaking_relevance", seg.overtaking_relevance),
        ):
            if not (0.0 <= value <= 1.0):
                raise ValueError(f"{field_name} out of range for {seg.name}")

        if i > 0:
            continuity_gap = abs(track[i - 1].v_exit_kph - seg.v_entry_kph)
            if continuity_gap > 7.5:
                raise ValueError(f"Speed continuity mismatch between {track[i - 1].name} and {seg.name}")


track = build_gilles_villeneuve_track()
validate_track(track)
print(f"Track loaded: {len(track)} segments, {sum(s.length_m for s in track):.0f} m")


In [ ]:
@dataclass(frozen=True)
class PolicyConfig:
    name: str
    base_scale: float
    reserve_soc: float
    priority_weight: float


def deployment_policy(segment: TrackSegment, soc: float, deploy_used_j: float, policy: PolicyConfig) -> float:
    if segment.segment_type != "accel":
        return 0.0
    if soc <= policy.reserve_soc:
        return 0.0

    energy_left = MGUK_MAX_DEPLOY_ENERGY_LAP - deploy_used_j
    if energy_left <= 0:
        return 0.0

    soc_factor = np.clip((soc - policy.reserve_soc) / (SOC_MAX - policy.reserve_soc), 0.0, 1.0)
    priority_factor = (1.0 - policy.priority_weight) + policy.priority_weight * segment.deployment_priority
    p_cmd = MGUK_MAX_DEPLOY_POWER * policy.base_scale * soc_factor * priority_factor

    return float(np.clip(p_cmd, 0.0, min(MGUK_MAX_DEPLOY_POWER, energy_left / DT)))


POLICIES = [
    PolicyConfig("Conservative", base_scale=0.75, reserve_soc=0.45, priority_weight=0.65),
    PolicyConfig("Balanced", base_scale=0.90, reserve_soc=0.35, priority_weight=0.75),
    PolicyConfig("Aggressive", base_scale=1.00, reserve_soc=0.28, priority_weight=0.85),
]


In [ ]:
def resistive_forces(speed_mps: float) -> float:
    drag = 0.5 * RHO_AIR * CD_A * speed_mps**2
    rolling = C_RR * VEHICLE_MASS * G
    return drag + rolling


def run_lap(track: List[TrackSegment], policy: PolicyConfig, ers_enabled: bool = True) -> Dict[str, np.ndarray]:
    soc = INITIAL_SOC
    deploy_used = 0.0
    regen_harvested = 0.0

    t = 0.0
    time_trace, soc_trace = [], []
    deploy_power_trace, regen_power_trace, speed_trace = [], [], []

    segment_deploy_j = np.zeros(len(track))
    segment_regen_j = np.zeros(len(track))
    segment_entry_speed_kph = np.zeros(len(track))
    segment_exit_speed_kph = np.zeros(len(track))

    for i, segment in enumerate(track):
        s_local = 0.0
        v = max(MIN_SPEED_MPS, segment.v_entry_mps)
        segment_entry_speed_kph[i] = v * 3.6

        while s_local < segment.length_m:
            f_resist = resistive_forces(v)
            p_deploy = 0.0
            p_regen = 0.0

            progress = np.clip(s_local / max(segment.length_m, 1e-6), 0.0, 1.0)
            v_target = segment.v_entry_mps + progress * (segment.v_exit_mps - segment.v_entry_mps)

            if segment.segment_type == "accel":
                f_ice = min(ICE_WHEEL_POWER / max(v, 1.0), TRACTION_FORCE_LIMIT)
                if ers_enabled:
                    p_deploy = deployment_policy(segment, soc, deploy_used, policy)
                f_mguk = (p_deploy * DEPLOY_EFFICIENCY) / max(v, 1.0)

                a_ctrl = 0.8 * (v_target - v)
                a = np.clip((f_ice + f_mguk - f_resist) / VEHICLE_MASS + a_ctrl, -2.0, 12.0)

            elif segment.segment_type == "brake":
                v_err = v - v_target
                a_des = np.clip(-0.8 * v_err - 1.5, -MAX_BRAKE_DECEL, -0.3)

                required_brake_force = max(0.0, VEHICLE_MASS * abs(a_des) + f_resist)
                recoverable_brake_force = 0.62 * required_brake_force

                if ers_enabled and soc < SOC_MAX - 0.01:
                    p_regen_theoretical = recoverable_brake_force * v
                    soc_acceptance = np.clip((SOC_MAX - soc) / 0.25, 0.0, 1.0)
                    p_regen = min(p_regen_theoretical, MGUK_MAX_REGEN_POWER) * soc_acceptance
                    e_harv = p_regen * REGEN_EFFICIENCY * DT
                    regen_harvested += e_harv
                    segment_regen_j[i] += e_harv
                    soc += e_harv / BATTERY_CAPACITY_J

                a = a_des
            else:
                a = -f_resist / VEHICLE_MASS

            if p_deploy > 0.0:
                e_dep = p_deploy * DT
                deploy_used += e_dep
                segment_deploy_j[i] += e_dep
                soc -= e_dep / BATTERY_CAPACITY_J

            soc = float(np.clip(soc, SOC_MIN, SOC_MAX))
            v = float(np.clip(v + a * DT, MIN_SPEED_MPS, 95.0))
            s_local += v * DT
            t += DT

            time_trace.append(t)
            soc_trace.append(soc)
            deploy_power_trace.append(p_deploy)
            regen_power_trace.append(p_regen)
            speed_trace.append(v)

        segment_exit_speed_kph[i] = v * 3.6

    return {
        "policy_name": policy.name,
        "lap_time_s": t,
        "deploy_used_j": deploy_used,
        "regen_harvested_j": regen_harvested,
        "time": np.array(time_trace),
        "soc": np.array(soc_trace),
        "deploy_power": np.array(deploy_power_trace),
        "regen_power": np.array(regen_power_trace),
        "speed": np.array(speed_trace),
        "segment_deploy_j": segment_deploy_j,
        "segment_regen_j": segment_regen_j,
        "segment_entry_speed_kph": segment_entry_speed_kph,
        "segment_exit_speed_kph": segment_exit_speed_kph,
    }


In [ ]:
def quality_checks(result: Dict[str, np.ndarray], label: str) -> Dict[str, float]:
    soc = result["soc"]
    speed_kph = result["speed"] * 3.6
    deploy = result["deploy_power"]
    regen = result["regen_power"]

    soc_est = INITIAL_SOC + (result["regen_harvested_j"] - result["deploy_used_j"]) / BATTERY_CAPACITY_J
    soc_error = abs(soc_est - soc[-1])

    checks = {
        "lap_time_s": float(result["lap_time_s"]),
        "soc_min": float(np.min(soc)),
        "soc_max": float(np.max(soc)),
        "speed_min_kph": float(np.min(speed_kph)),
        "speed_max_kph": float(np.max(speed_kph)),
        "deploy_cap_violation_w": float(np.max(np.maximum(deploy - MGUK_MAX_DEPLOY_POWER, 0.0))),
        "regen_cap_violation_w": float(np.max(np.maximum(regen - MGUK_MAX_REGEN_POWER, 0.0))),
        "energy_cap_margin_j": float(MGUK_MAX_DEPLOY_ENERGY_LAP - result["deploy_used_j"]),
        "soc_balance_error": float(soc_error),
    }

    print(f"\n--- Quality checks: {label} ---")
    print(f"Lap time              : {checks['lap_time_s']:.2f} s")
    print(f"SOC range             : {checks['soc_min']:.3f} to {checks['soc_max']:.3f}")
    print(f"Speed range           : {checks['speed_min_kph']:.1f} to {checks['speed_max_kph']:.1f} km/h")
    print(f"Deploy cap overrun    : {checks['deploy_cap_violation_w']:.2f} W")
    print(f"Regen cap overrun     : {checks['regen_cap_violation_w']:.2f} W")
    print(f"Deploy energy margin  : {checks['energy_cap_margin_j']/1e6:.3f} MJ")
    print(f"SOC balance error     : {checks['soc_balance_error']:.6f}")

    return checks


def calibration_report(checks: Dict[str, float], label: str) -> None:
    tests = [
        ("Lap time band", CAL_TARGETS['lap_time_min_s'] <= checks['lap_time_s'] <= CAL_TARGETS['lap_time_max_s']),
        ("Top speed band", CAL_TARGETS['top_speed_min_kph'] <= checks['speed_max_kph'] <= CAL_TARGETS['top_speed_max_kph']),
        ("Min speed band", CAL_TARGETS['min_speed_min_kph'] <= checks['speed_min_kph'] <= CAL_TARGETS['min_speed_max_kph']),
        ("Deploy cap obeyed", checks['deploy_cap_violation_w'] <= 1e-6),
        ("Regen cap obeyed", checks['regen_cap_violation_w'] <= 1e-6),
        ("Deploy energy <= 4 MJ", checks['energy_cap_margin_j'] >= -1e-6),
        ("SOC balance close", checks['soc_balance_error'] <= CAL_TARGETS['max_soc_balance_error']),
    ]

    print(f"\n=== Phase 1A Calibration Report: {label} ===")
    for name, ok in tests:
        status = "PASS" if ok else "WARN"
        print(f"{status:<5} | {name}")


def print_segment_speed_report(track: List[TrackSegment], result: Dict[str, np.ndarray], title: str) -> None:
    print(f"\n=== Segment Speed Report: {title} ===")
    print("Segment                          | Entry Sim | Exit Sim | Exit Target | Exit Error")
    print("-" * 82)
    for i, seg in enumerate(track):
        e_in = result['segment_entry_speed_kph'][i]
        e_out = result['segment_exit_speed_kph'][i]
        tgt = seg.v_exit_kph
        err = e_out - tgt
        print(f"{seg.name[:30]:<32} | {e_in:>8.1f} | {e_out:>8.1f} | {tgt:>11.1f} | {err:>9.1f}")


def moving_average(x: np.ndarray, window: int = 25) -> np.ndarray:
    if window <= 1:
        return x.copy()
    kernel = np.ones(window) / window
    return np.convolve(x, kernel, mode="same")


def run_policy_comparison(track: List[TrackSegment], policies: List[PolicyConfig]):
    baseline_policy = PolicyConfig("ERS_OFF_REF", 0.0, 0.99, 0.0)
    baseline = run_lap(track, baseline_policy, ers_enabled=False)

    results = []
    print("=== Policy Comparison Summary ===")
    print("Policy         | Lap Time (s) | Gain vs OFF (s) | Deploy (MJ) | Harvest (MJ) | Final SOC")
    print("-"*84)
    for p in policies:
        r = run_lap(track, p, ers_enabled=True)
        gain = baseline['lap_time_s'] - r['lap_time_s']
        print(f"{p.name:<14}| {r['lap_time_s']:>11.2f} | {gain:>14.2f} | {r['deploy_used_j']/1e6:>10.2f} | {r['regen_harvested_j']/1e6:>11.2f} | {r['soc'][-1]:>8.3f}")
        results.append(r)

    return baseline, results


baseline_result, policy_results = run_policy_comparison(track, POLICIES)
selected_result = next(r for r in policy_results if r['policy_name'] == 'Balanced')

print("\n=== Detailed Summary (Balanced Policy) ===")
print(f"Baseline lap time (ERS OFF): {baseline_result['lap_time_s']:.2f} s")
print(f"Hybrid lap time   (ERS ON) : {selected_result['lap_time_s']:.2f} s")
print(f"Estimated lap-time gain    : {baseline_result['lap_time_s'] - selected_result['lap_time_s']:.2f} s")
print(f"Energy deployed            : {selected_result['deploy_used_j']/1e6:.2f} MJ / 4.00 MJ")
print(f"Energy harvested           : {selected_result['regen_harvested_j']/1e6:.2f} MJ")
print(f"Final SOC                  : {selected_result['soc'][-1]:.3f}")

baseline_checks = quality_checks(baseline_result, "ERS OFF")
balanced_checks = quality_checks(selected_result, "Balanced")
calibration_report(baseline_checks, "ERS OFF")
calibration_report(balanced_checks, "Balanced")
print_segment_speed_report(track, selected_result, "Balanced")


In [ ]:
# Plot 1: Battery SOC trace
result = selected_result
plt.figure(figsize=(10, 4))
plt.plot(result["time"], result["soc"], label="SOC")
plt.title("Battery SOC vs Time - Circuit Gilles Villeneuve (Balanced Policy)")
plt.xlabel("Time (s)")
plt.ylabel("SOC")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:
# Plot 2: MGU-K power trace (raw + smoothed)
result = selected_result
deploy_kw = result["deploy_power"] / 1000.0
regen_kw = result["regen_power"] / 1000.0

plt.figure(figsize=(10, 5))
plt.plot(result["time"], deploy_kw, alpha=0.35, label="Deploy Raw (kW)")
plt.plot(result["time"], regen_kw, alpha=0.35, label="Regen Raw (kW)")
plt.plot(result["time"], moving_average(deploy_kw, 25), linewidth=2.2, label="Deploy Smoothed (kW)")
plt.plot(result["time"], moving_average(regen_kw, 25), linewidth=2.2, label="Regen Smoothed (kW)")
plt.title("MGU-K Power vs Time (Raw + Smoothed)")
plt.xlabel("Time (s)")
plt.ylabel("Power (kW)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:
# Plot 3: Vehicle speed trace
result = selected_result
plt.figure(figsize=(10, 4))
plt.plot(result["time"], result["speed"] * 3.6)
plt.title("Vehicle Speed Trace (Balanced Policy)")
plt.xlabel("Time (s)")
plt.ylabel("Speed (km/h)")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Plot 4: Policy-level lap-time comparison
policy_names = [r['policy_name'] for r in policy_results]
lap_times = [r['lap_time_s'] for r in policy_results]
gains = [baseline_result['lap_time_s'] - r['lap_time_s'] for r in policy_results]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(policy_names, lap_times)
ax[0].set_title("Lap Time by Policy")
ax[0].set_ylabel("Lap Time (s)")
ax[0].grid(True, alpha=0.25)

ax[1].bar(policy_names, gains)
ax[1].set_title("Lap-Time Gain vs ERS OFF")
ax[1].set_ylabel("Gain (s)")
ax[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:
# Plot 5: Segment-level energy map
result = selected_result
segment_names = [s.name for s in track]
x = np.arange(len(track))
width = 0.4

plt.figure(figsize=(12, 4))
plt.bar(x - width/2, result['segment_deploy_j']/1e6, width=width, label='Deploy (MJ)')
plt.bar(x + width/2, result['segment_regen_j']/1e6, width=width, label='Regen (MJ)')
plt.xticks(x, segment_names, rotation=25, ha='right')
plt.title('Segment Energy Map (Balanced Policy)')
plt.ylabel('Energy (MJ)')
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()
